# Aggregate Yearly Data To Make Inital DataFrame  

In [ ]:
import pandas as pd
import os

archive_path = 'archive'
dfs = {}

for f in sorted(os.listdir(archive_path)):
    df = pd.read_csv(os.path.join(archive_path, f))
    dfs[f] = df

rows = []
for filename, df in dfs.items():
    year = int(filename.replace('.csv', ''))
    top10 = df.nlargest(10, 'Weight')[['Company', 'Ticker', 'Weight']].reset_index(drop=True)
    row = {'Year': year}
    for i, r in top10.iterrows():
        row[f'Company_{i+1}'] = r['Company']
        row[f'Ticker_{i+1}'] = r['Ticker']
        row[f'Weight_{i+1}'] = r['Weight']
    rows.append(row)

top10_df = pd.DataFrame(rows).sort_values('Year').reset_index(drop=True)
# top10_df.to_csv('top10.csv')
# print(top10_df)

## Getting & Aggregating Monthly Data

In [ ]:
selected_cols = ['Ticker_1', 'Ticker_2', 'Ticker_3', 'Ticker_4', 'Ticker_5', 'Ticker_6', 'Ticker_7', 'Ticker_8', 'Ticker_9', 'Ticker_10']

unique_array = pd.unique(top10_df[selected_cols].values.ravel())
unique_list = unique_array.tolist()

import yfinance as yf
import pandas as pd
import os

top10_df = pd.read_csv('top10.csv')

long_records = []
for _, row in top10_df.iterrows():
    year = int(row['Year'])
    for rank in range(1, 11):
        long_records.append({
            'Year': year,
            'Rank': rank,
            'Ticker': row[f'Ticker_{rank}'],
            'Company': row[f'Company_{rank}'],
            'Weight': row[f'Weight_{rank}'],
        })

weight_df = pd.DataFrame(long_records)   # 260 rows × 5 cols
# print(f"weight_df shape: {weight_df.shape}")
# print(weight_df.head(12).to_string())

from tqdm.notebook import tqdm

tickers = [
    'GE','CSCO','MSFT','XOM','PFE','INTC','C','ORCL','AIG','DELL',
    'WMT','JNJ','IBM','T','KO','PG','BAC','MO','JPM','CVX',
    'AAPL','GOOGL','BRK-B','WFC','META','AMZN','GOOG','V','TSLA','NVDA',
    'UNH','AVGO'
]
TICKER_MAP_REVERSE = {'BRK-B': 'BRK.B'} 

price_frames = []
for ticker in tqdm(tickers):
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(period='max', interval='1mo')
        if df.empty:
            print(f"  {ticker}: NO DATA"); continue

        df = df.reset_index()
        df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
        df['Date'] = df['Date'].dt.to_period('M').dt.to_timestamp()

        keep = ['Date','Open','High','Low','Close','Volume','Dividends']
        df = df[[c for c in keep if c in df.columns]].copy()
        df['Ticker'] = TICKER_MAP_REVERSE.get(ticker, ticker)
        price_frames.append(df)
        print(f"  {ticker}: {len(df)} rows")
    except Exception as e:
        print(f"  {ticker}: ERROR {e}")


all_prices = pd.concat(price_frames, ignore_index=True)
all_prices['Year'] = all_prices['Date'].dt.year
all_prices = all_prices[(all_prices['Year'] >= 2000) & (all_prices['Year'] <= 2025)]
monthly_df = all_prices.merge(weight_df, on=['Ticker', 'Year'], how='inner')
col_order = ['Date','Year','Rank','Ticker','Company','Open','High','Low','Close','Volume','Dividends','Weight']
monthly_df = monthly_df[col_order]
monthly_df = monthly_df.sort_values(['Year','Rank','Date']).reset_index(drop=True)
monthly_df.to_csv('top10_monthly.csv', index=False)
# print(f"\nSaved → top10_monthly.csv  ({monthly_df.shape[0]} rows × {monthly_df.shape[1]} cols)")

# ...

# STILL NEED TO ADD INTERMED STEPS

# ...

In [1]:
"""
=============================================================================
  CAUSAL DISCOVERY & INFERENCE — S&P 500 Top-10 Monthly Holdings (2000–2025)
=============================================================================

OUTLINE
-------
0.  Setup & Data Loading
1.  Feature Engineering (aggregate time-series variables)
2.  Stationarity Testing (ADF)
3.  Independence in Causal Models  (correlation + partial correlation)
4.  Constraint-Based Causal Discovery
      4a. PC Algorithm (implemented from scratch with Fisher-Z CI tests)
      4b. FCI Algorithm (skeleton + orientation + possible-ancestry rules)
5.  Granger Causality (F-test via OLS, pairwise + multivariate)
6.  PCMCI  (PC-MCI for time-series, lag-aware causal graph)
7.  Causal Graph Visualization  (using networkx + matplotlib)
8.  Summary of Findings

Variables used
--------------
  tech_ret     : weighted-average monthly return of Tech top-10 holdings
  nontech_ret  : weighted-average monthly return of Non-Tech top-10 holdings
  sp500_ret    : equal proxy = average of all top-10 holdings monthly return
  top10_conc   : monthly concentration (Herfindahl) of top-10 weights
  tech_weight  : fraction of top-10 weight held by Tech stocks
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import networkx as nx

from itertools import combinations, permutations
from scipy import stats
from scipy.linalg import lstsq

np.random.seed(28)

print("Loading data and engineering features")

main_df = pd.read_csv(
    "full_stock_s&p500_weights_monthly.csv",
    parse_dates=["Date"]
)

print(f"main_df.columns:\n{main_df.columns.tolist()}\n")
# print(f"main_df.head(20):\n{main_df.head(20)}\n")
# print(f"main_df.tail(20):\n{main_df.tail(20)}\n")

Loading data and engineering features
main_df.columns:
['Unnamed: 0', 'Date', 'Year', 'Rank', 'Ticker', 'Company', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Weight', 'Sector', 'Era', 'NormClose']



In [2]:
TECH = {
    "MSFT","AAPL","CSCO","INTC","ORCL","IBM","AMZN",
    "META","GOOG","GOOGL","NVDA","AVGO","TSLA","DELL"
}
main_df["Sector"] = main_df["Ticker"].apply(
    lambda x: "Tech" if x in TECH else "Non-Tech"
)

# applying the monthly log-return per ticker
main_df = main_df.sort_values(["Ticker", "Date"])
main_df["LogRet"] = main_df.groupby("Ticker")["Close"].transform(lambda x: np.log(x / x.shift(1)))

# aggregate to one row per month
def wavg(df, val_col, wt_col):
    w = df[wt_col].values
    v = df[val_col].values
    mask = ~np.isnan(v)
    if mask.sum() == 0 or w[mask].sum() == 0:
        return np.nan
    return np.average(v[mask], weights=w[mask])

monthly_agg = ( # W Claude :)
    main_df.groupby("Date")
    .apply(lambda g: pd.Series({
        "sp500_ret" : wavg(g.dropna(subset=["LogRet"]), "LogRet", "Weight"),
        "tech_ret" : wavg(g[g["Sector"]=="Tech"].dropna(subset=["LogRet"]), "LogRet", "Weight"),
        "nontech_ret" : wavg(g[g["Sector"]=="Non-Tech"].dropna(subset=["LogRet"]), "LogRet", "Weight"),
        "tech_weight" : g[g["Sector"]=="Tech"]["Weight"].sum() / g["Weight"].sum() if g["Weight"].sum()>0 else np.nan,
        "top10_conc" : (g["Weight"]**2).sum() / (g["Weight"].sum()**2) if g["Weight"].sum()>0 else np.nan,
    }))
    .reset_index()
    .sort_values("Date")
)

# drop rows missing any variable -> mainly first row per ticker
monthly_agg.dropna(inplace=True)
monthly_agg.reset_index(drop=True, inplace=True)

print(f"Monthly aggregate shape: {monthly_agg.shape}")
print(f"Date range: {monthly_agg['Date'].min().date()} -> {monthly_agg['Date'].max().date()}\n")
print(f"Columns: {list(monthly_agg.columns)}")
print(f"Head:\n{monthly_agg.head(5).to_string()}")

VARS = ["tech_ret", "nontech_ret", "sp500_ret", "top10_conc", "tech_weight"]
VAR_LABELS = {
    "tech_ret" : "Tech\nReturn",
    "nontech_ret" : "Non-Tech\nReturn",
    "sp500_ret" : "S&P500\nReturn",
    "top10_conc" : "Top-10\nConc.",
    "tech_weight" : "Tech\nWeight",
}
data_mat = monthly_agg[VARS].values.astype(float)
T, N = data_mat.shape

print(f"\nVariables ({N}) X Time-steps ({T}) = {N * T}")

Monthly aggregate shape: (311, 6)
Date range: 2000-02-01 -> 2025-12-01

Columns: ['Date', 'sp500_ret', 'tech_ret', 'nontech_ret', 'tech_weight', 'top10_conc']
Head:
        Date  sp500_ret  tech_ret  nontech_ret  tech_weight  top10_conc
0 2000-02-01   0.013828  0.140424    -0.080156     0.426076    0.123074
1 2000-03-01   0.139682  0.141118     0.138616     0.426076    0.123074
2 2000-04-01  -0.048826 -0.150488     0.026647     0.426076    0.123074
3 2000-05-01  -0.028633 -0.115681     0.035992     0.426076    0.123074
4 2000-06-01   0.065708  0.144964     0.006869     0.426076    0.123074

Variables (5) X Time-steps (311) = 1555


In [3]:
print("Stationarity Testing with Augmented Dickey-Fuller (ADF) Test") # read this for more info -> https://en.wikipedia.org/wiki/Augmented_Dickey%E2%80%93Fuller_test
def adf_test_manual(series, maxlag=4): # W Claude :)
    """
    Manual ADF test using OLS where H0: unit root (non-stationary) & we Reject H0 if t-stat < critical value.
    Returns (t_stat, approx_pvalue).
    Critical values (MacKinnon 1994) for n->infinity: -3.43 (1%), -2.86 (5%), -2.57 (10%).
    We approximate p-values via interpolation on these three points.
    """
    y = np.array(series, dtype=float)
    dy = np.diff(y)
    n = len(dy)
    # build regression matrix: [y_{t-1}, Δy_{t-1}, …, Δy_{t-p}, 1]
    # use simple fixed-window construction
    # dependent variable: dy[maxlag:]
    dy2 = dy[maxlag:]
    n2 = len(dy2)
    # regressors: y_{t-1} = y[maxlag:-1] or y[maxlag:maxlag+n2]
    cols = [y[maxlag: maxlag+n2]] # y_{t-1}
    for lag in range(1, maxlag+1):
        cols.append(dy[maxlag-lag: maxlag-lag+n2])
    cols.append(np.ones(n2))
    min_len = min(len(c) for c in cols)
    dy2 = dy2[:min_len]
    X = np.column_stack([c[:min_len] for c in cols])
    # OLS
    coef, res, _, _ = lstsq(X, dy2)
    resid = dy2 - X @ coef
    sigma2 = (resid @ resid) / max(1, min_len - X.shape[1])
    se = np.sqrt(sigma2 * np.diag(np.linalg.pinv(X.T @ X)))
    t_stat = coef[0] / se[0] if se[0] > 0 else 0.0
    # Approximate p-value: linear interp on MacKinnon critical values
    cv = np.array([-3.43, -2.86, -2.57])
    pv = np.array([0.01,   0.05,  0.10])
    if t_stat < cv[0]:
        p = 0.005
    elif t_stat > cv[-1]:
        p = 0.20
    else:
        p = float(np.interp(t_stat, cv, pv))
    return t_stat, p

print(f"\n{'Variable':<16} {'ADF t-stat':>12} {'p-value':>10} {'Stationary?':>14}")
print(f"{'-'*16} {'-'*12} {'-'*10} {'-'*14}")
adf_results = {}
for var in VARS:
    t, p = adf_test_manual(monthly_agg[var].values)
    stat = "Yes (5%)" if p < 0.05 else "No"
    print(f"{var:<16} {t:>12.4f} {p:>10.4f} {stat:>14}")
    adf_results[var] = (t, p)

# first difference the potentially non-stationary series for causal models
monthly_agg["d_tech_weight"] = monthly_agg["tech_weight"].diff()
monthly_agg["d_top10_conc"]  = monthly_agg["top10_conc"].diff()

# build the stationary dataset used by all causal methods
df_stat = monthly_agg.dropna().copy().reset_index(drop=True)
VARS_STAT = ["tech_ret", "nontech_ret", "sp500_ret", "d_top10_conc", "d_tech_weight"]
VAR_LABELS_STAT = {
    "tech_ret" : "Tech\nReturn",
    "nontech_ret" : "Non-Tech\nReturn",
    "sp500_ret" : "S&P500\nReturn",
    "d_top10_conc" : "ΔTop-10\nConc.",
    "d_tech_weight" : "ΔTech\nWeight",
}
X_stat = df_stat[VARS_STAT].values.astype(float)
T_s, N_s = X_stat.shape
print(f"Stationary dataset shape: {T_s} x {N_s}")


Stationarity Testing with Augmented Dickey-Fuller (ADF) Test

Variable           ADF t-stat    p-value    Stationary?
---------------- ------------ ---------- --------------
tech_ret              -8.3030     0.0050       Yes (5%)
nontech_ret           -7.8240     0.0050       Yes (5%)
sp500_ret             -7.4353     0.0050       Yes (5%)
top10_conc            -1.5629     0.2000             No
tech_weight            0.0512     0.2000             No
Stationary dataset shape: 310 x 5


In [4]:
print("Independence in Causal Models?")

# Pearson Correlations
corr_mat = np.corrcoef(X_stat.T)
print("\nPearson Correlation Matrix:")
header = f"{'':>18}" + "".join(f"{v:>18}" for v in VARS_STAT)
print(header)
for i, vi in enumerate(VARS_STAT):
    row = f"{vi:>18}" + "".join(f"{corr_mat[i,j]:>18.4f}" for j in range(N_s))
    print(row)

def partial_corr_matrix(X):  # cool read -> https://arxiv.org/html/2502.08414v1
    #Partial correlation via precision matrix (inverse of correlation)
    C = np.corrcoef(X.T)
    try:
        P = np.linalg.inv(C)
    except np.linalg.LinAlgError:
        P = np.linalg.pinv(C)
    n = P.shape[0]
    PC = np.zeros_like(P)
    for i in range(n):
        for j in range(n):
            PC[i, j] = -P[i, j] / np.sqrt(max(P[i,i]*P[j,j], 1e-12))
    np.fill_diagonal(PC, 1.0)
    return PC

pcorr_mat = partial_corr_matrix(X_stat)
print("\nPartial Correlation Matrix (conditioning on all others):")
print(header)
for i, vi in enumerate(VARS_STAT):
    row = f"  {vi:>18}" + "".join(f"{pcorr_mat[i,j]:>18.4f}" for j in range(N_s))
    print(row)

# Fisher-Z conditional independence tests  
def fisher_z_test(X, i, j, cond_set, alpha=0.05): # https://causal-learn.readthedocs.io/en/latest/independence_tests_index/index.html
    """
    Fisher-Z test for conditional independence of X[:,i] ⊥ X[:,j] | X[:,cond_set].
    Returns (is_independent, p_value, partial_corr).
    """
    n = X.shape[0]
    if len(cond_set) == 0:
        r = np.corrcoef(X[:, i], X[:, j])[0, 1]
    else:
        # partial corr via regression residuals
        Z  = np.column_stack([X[:, k] for k in cond_set] + [np.ones(n)])
        ri = X[:, i] - Z @ lstsq(Z, X[:, i])[0]
        rj = X[:, j] - Z @ lstsq(Z, X[:, j])[0]
        denom = np.sqrt(np.sum(ri**2) * np.sum(rj**2))
        r = np.dot(ri, rj) / denom if denom > 0 else 0.0
    r = np.clip(r, -0.9999, 0.9999)
    z = 0.5 * np.log((1 + r) / (1 - r))
    se= 1.0 / np.sqrt(max(n - len(cond_set) - 3, 1))
    t = z / se
    p = 2 * (1 - stats.norm.cdf(abs(t)))
    return p >= alpha, p, r

print("\nSelected Conditional Independence Tests (Fisher-Z, α=0.05):")
pairs_to_test = [
    (0, 2, [1]), # tech_ret ⊥ sp500_ret | nontech_ret
    (0, 1, [2]),# tech_ret ⊥ nontech_ret | sp500_ret
    (0, 4, []), # tech_ret ⊥ d_tech_weight
    (0, 4, [1, 2]), # tech_ret ⊥ d_tech_weight | nontech_ret, sp500_ret
    (1, 4, [0, 2]), # nontech_ret ⊥ d_tech_weight | tech_ret, sp500_ret
    (2, 3, [0, 1]), # sp500_ret ⊥ d_top10_conc | tech_ret, nontech_ret
]
print(f"\n{'Test':<55} {'p-value':>8} {'indep?':>8} {'pcorr':>8}")
print(f"{'-'*55} {'-'*8} {'-'*8} {'-'*8}")
ci_results = {}
for i, j, cond in pairs_to_test:
    vi, vj = VARS_STAT[i], VARS_STAT[j]
    cond_names = [VARS_STAT[k] for k in cond]
    label = f"{vi} ⊥ {vj}" + (f" | {','.join(cond_names)}" if cond else "")
    indep, pv, pc = fisher_z_test(X_stat, i, j, cond)
    ci_results[(i,j,tuple(cond))] = (indep, pv, pc)
    print(f"  {label:<55} {pv:>8.4f} {'Yes' if indep else 'No':>8} {pc:>8.4f}")

Independence in Causal Models?

Pearson Correlation Matrix:
                            tech_ret       nontech_ret         sp500_ret      d_top10_conc     d_tech_weight
          tech_ret            1.0000            0.3757            0.8291           -0.0380           -0.1779
       nontech_ret            0.3757            1.0000            0.6594            0.0753           -0.1274
         sp500_ret            0.8291            0.6594            1.0000            0.0703           -0.0847
      d_top10_conc           -0.0380            0.0753            0.0703            1.0000            0.2552
     d_tech_weight           -0.1779           -0.1274           -0.0847            0.2552            1.0000

Partial Correlation Matrix (conditioning on all others):
                            tech_ret       nontech_ret         sp500_ret      d_top10_conc     d_tech_weight
            tech_ret            1.0000           -0.4314            0.8447           -0.1110           -0.2205
        

In [6]:
print("PC Algorithm (constraint-based causal discovery)")

def pc_algorithm(X, var_names, alpha=0.05):
    """
    Peter-Clark (PC) algorithm for causal skeleton + orientation.
    Uses Fisher-Z conditional independence tests.
    Returns: (skeleton graph, separation sets, CPDAG as nx.DiGraph)
    """
    n_vars  = X.shape[1]
    # complete undirected graph
    adj = {i: set(range(n_vars)) - {i} for i in range(n_vars)}
    sep_set = {(i,j): None for i in range(n_vars) for j in range(n_vars) if i!=j}

    print(f"\nSkeleton learning (alpha={alpha})")
    cond_size = 0
    while True:
        removed = False
        for i in range(n_vars):
            for j in list(adj[i]):
                # find conditioning sets of size cond_size from adj[i] \ {j}
                candidates = list(adj[i] - {j})
                if len(candidates) < cond_size:
                    continue
                for cond in combinations(candidates, cond_size):
                    indep, pv, _ = fisher_z_test(X, i, j, list(cond), alpha)
                    if indep:
                        adj[i].discard(j)
                        adj[j].discard(i)
                        sep_set[(i,j)] = list(cond)
                        sep_set[(j,i)] = list(cond)
                        removed = True
                        print(f"Removed edge {var_names[i]}—{var_names[j]} | cond={[var_names[c] for c in cond]} (p={pv:.4f})")
                        break
        cond_size += 1
        if cond_size > n_vars - 2:
            break
        if not removed and cond_size > 1:
            break # no more edges to remove at this or larger cond size

    # skeleton as undirected graph
    G = nx.Graph()
    G.add_nodes_from(range(n_vars))
    for i in range(n_vars):
        for j in adj[i]:
            if i < j:
                G.add_edge(i, j)

    print(f"\nSkeleton edges ({G.number_of_edges()}):")
    for u, v in G.edges():
        print(f"{var_names[u]} - {var_names[v]}")

    # orient v-structures (colliders)
    DG = nx.DiGraph()
    DG.add_nodes_from(range(n_vars))
    # start with all skeleton edges as bidirected
    for u, v in G.edges():
        DG.add_edge(u, v)
        DG.add_edge(v, u)

    oriented = set()
    print("\nv-structure orientation (colliders):")
    for b in range(n_vars):
        nbrs = list(adj[b])
        for a, c in combinations(nbrs, 2):
            if c not in adj[a]:
                s_ac = sep_set.get((a,c), [])
                if s_ac is not None and b not in s_ac:
                    if DG.has_edge(b, a): DG.remove_edge(b, a)
                    if DG.has_edge(b, c): DG.remove_edge(b, c)
                    oriented.add((a,b))
                    oriented.add((c,b))
                    print(f"V-structure: {var_names[a]} → {var_names[b]} ← {var_names[c]}")

    # meek's orientation rules (R1–R3 simplified)
    def apply_meek_rules(DG, adj, n_vars):
        changed = True
        while changed:
            changed = False
            for a in range(n_vars):
                for b in adj[a]:
                    if DG.has_edge(a,b) and DG.has_edge(b,a):
                        for c in range(n_vars):
                            if c!=a and c!=b:
                                if (DG.has_edge(c,a) and not DG.has_edge(a,c) and c not in adj[b]):
                                    DG.remove_edge(b, a)
                                    changed = True
                                    break
        return DG

    DG = apply_meek_rules(DG, adj, n_vars)

    # build final CPDAG: keep only unambiguous directed edges
    CPDAG = nx.DiGraph()
    CPDAG.add_nodes_from(range(n_vars))
    for u, v in DG.edges():
        if not DG.has_edge(v, u):   # directed
            CPDAG.add_edge(u, v, style="directed")
        elif (u, v) not in [(e[1],e[0]) for e in CPDAG.edges()]:
            CPDAG.add_edge(u, v, style="undirected")   # undirected — add once

    return G, sep_set, CPDAG, DG

short_names = ["tech_ret", "nontech_ret", "sp500_ret", "d_conc", "d_tw"]
G_skel, sep_sets, CPDAG, DG_pc = pc_algorithm(X_stat, short_names, alpha=0.05)

print("\nPC Algorithm — Final CPDAG edges:")
for u, v, d in CPDAG.edges(data=True):
    style = d.get("style","?")
    arrow = "->" if style=="directed" else "-"
    print(f"{short_names[u]} {arrow} {short_names[v]}")

PC Algorithm (constraint-based causal discovery)

Skeleton learning (alpha=0.05)
Removed edge tech_ret—d_conc | cond=[] (p=0.5057)
Removed edge nontech_ret—d_conc | cond=[] (p=0.1860)
Removed edge sp500_ret—d_conc | cond=[] (p=0.2176)
Removed edge sp500_ret—d_tw | cond=[] (p=0.1370)
Removed edge nontech_ret—d_tw | cond=['tech_ret'] (p=0.2447)

Skeleton edges (5):
tech_ret - nontech_ret
tech_ret - sp500_ret
tech_ret - d_tw
nontech_ret - sp500_ret
d_conc - d_tw

v-structure orientation (colliders):
V-structure: sp500_ret → tech_ret ← d_tw
V-structure: tech_ret → d_tw ← d_conc

PC Algorithm — Final CPDAG edges:
tech_ret - nontech_ret
nontech_ret - sp500_ret
sp500_ret -> tech_ret
d_conc -> d_tw


In [7]:
print("FCI Algorithm (handles latent confounders / selection bias)")

def fci_algorithm(X, var_names, alpha=0.05):
    """
    Fast Causal Inference (FCI) - simplified implementation.
    Phase 1: Same skeleton as PC.
    Phase 2: Marks arrowheads for potential latent confounders.
    Returns PAG (Partial Ancestral Graph) as nx.DiGraph with edge types.
    """
    n_vars = X.shape[1]
    adj = {i: set(range(n_vars)) - {i} for i in range(n_vars)}
    sep_set = {(i,j): None for i,j in [(a,b) for a in range(n_vars) for b in range(n_vars) if a!=b]}

    print(f"\nFCI Skeleton (same CI tests as PC, alpha={alpha})")
    cond_size = 0
    while True:
        removed = False
        for i in range(n_vars):
            for j in list(adj[i]):
                cands = list(adj[i] - {j})
                if len(cands) < cond_size:
                    continue
                for cond in combinations(cands, cond_size):
                    indep, pv, _ = fisher_z_test(X, i, j, list(cond), alpha)
                    if indep:
                        adj[i].discard(j); adj[j].discard(i)
                        sep_set[(i,j)] = list(cond)
                        sep_set[(j,i)] = list(cond)
                        removed = True
                        break
        cond_size += 1
        if cond_size > n_vars-2 or (not removed and cond_size > 1):
            break

    # PAG: edge marks are 'tail'(-), 'arrowhead'(>), 'circle'(o)
    # Represent as: edge_marks[(i,j)] = mark at j end of edge i→j
    edge_marks = {}
    for i in range(n_vars):
        for j in adj[i]:
            edge_marks[(i,j)] = "circle" # start with o—o (all circles)

    print("\nFCI V-structures & Possible Ancestral Edges")

    def is_possible_parent(i, j, adj, sep_set, n_vars):
        """i is a possible parent of j if i is adjacent to j and
           for no subset S of adj(i) is {i⊥j|S} established."""
        return j in adj[i]

    # orient definite non-colliders and v-structures
    pag_arrowheads = set()  # (i,j) means arrowhead at j
    pag_tails = set()  # (i,j) means tail at j
    for b in range(n_vars):
        nbrs = list(adj[b])
        for a, c in combinations(nbrs, 2):
            if c not in adj[a]:
                s_ac = sep_set.get((a,c), [])
                if s_ac is not None and b not in s_ac:
                    # a *→ b ←* c
                    pag_arrowheads.add((a, b)) # arrowhead at b from a
                    pag_arrowheads.add((c, b)) # arrowhead at b from c
                    print(f"    Definite collider: {var_names[a]} *→ {var_names[b]} ←* {var_names[c]}")
    changed = True
    while changed:
        changed = False
        for b in range(n_vars):
            for a in adj[b]:
                if (a,b) in pag_arrowheads: # a *→ b
                    for c in adj[b]:
                        if c == a: continue
                        if c not in adj[a]: # a not adj c
                            if (b,c) not in pag_arrowheads and (b,c) not in pag_tails:
                                pag_tails.add((c, b)) # tail at b
                                pag_arrowheads.add((b, c)) # arrowhead at c
                                changed = True
                                print(f"R1: {var_names[a]} *-> {var_names[b]} -> {var_names[c]}")

    # build PAG for output
    PAG = nx.DiGraph()
    PAG.add_nodes_from(range(n_vars))
    for i in range(n_vars):
        for j in adj[i]:
            if i < j:
                ij_head = (i,j) in pag_arrowheads
                ji_head = (j,i) in pag_arrowheads
                ij_tail = (i,j) in pag_tails
                ji_tail = (j,i) in pag_tails
                if ij_head and ji_head:
                    etype = "bidirected" 
                elif ij_head and ji_tail:
                    etype = "i<-j"
                elif ji_head and ij_tail:
                    etype = "i->j"
                else:
                    etype = "undirected"
                PAG.add_edge(i, j, etype=etype)

    print("\nFCI PAG edges:")
    for u, v, d in PAG.edges(data=True):
        et = d.get("etype","?")
        if et == "bidirected":
            sym = f"{var_names[u]} ↔ {var_names[v]}  (latent confounder)"
        elif et == "i->j":
            sym = f"{var_names[u]} → {var_names[v]}"
        elif et == "i<-j":
            sym = f"{var_names[u]} ← {var_names[v]}"
        else:
            sym = f"{var_names[u]} — {var_names[v]}  (undirected)"
        print(f"{sym}")
    return PAG

PAG = fci_algorithm(X_stat, short_names, alpha=0.05)

FCI Algorithm (handles latent confounders / selection bias)

FCI Skeleton (same CI tests as PC, alpha=0.05)

FCI V-structures & Possible Ancestral Edges
    Definite collider: sp500_ret *→ tech_ret ←* d_tw
    Definite collider: tech_ret *→ d_tw ←* d_conc
R1: d_tw *-> tech_ret -> nontech_ret
R1: d_tw *-> tech_ret -> sp500_ret
R1: tech_ret *-> d_tw -> d_conc

FCI PAG edges:
tech_ret ← nontech_ret
tech_ret ↔ sp500_ret  (latent confounder)
tech_ret ↔ d_tw  (latent confounder)
nontech_ret — sp500_ret  (undirected)
d_conc ↔ d_tw  (latent confounder)


In [8]:
print("Granger Causality (pairwise + multivariate)")

def granger_test(y, x, max_lag=6): # W Claude :) 
    """
    Pairwise Granger causality: does x Granger-cause y?
    Restricted model:  y_t = Σ a_i y_{t-i} + ε
    Unrestricted model: y_t = Σ a_i y_{t-i} + Σ b_i x_{t-i} + ε
    Returns dict of {lag: (F_stat, p_value)} for lag 1..max_lag.
    """
    results = {}
    for lag in range(1, max_lag+1):
        n = len(y)
        if n <= 2*lag + 2:
            continue
        # build lagged matrices
        # restricted: [y_{t-1}, …, y_{t-lag}, 1]
        ys = np.array([y[lag-k:-k if k>0 else n] for k in range(1, lag+1)]).T
        xs = np.array([x[lag-k:-k if k>0 else n] for k in range(1, lag+1)]).T
        yn = y[lag:]
        ones = np.ones((len(yn), 1))
        mn = min(len(yn), ys.shape[0], xs.shape[0])
        yn = yn[:mn]
        ys = ys[:mn]
        xs = xs[:mn]
        ones_r = ones[:mn]
        Xr = np.hstack([ys, ones_r])
        Xu = np.hstack([ys, xs, ones_r])
        # OLS
        coef_r, _, _, _ = lstsq(Xr, yn)
        coef_u, _, _, _ = lstsq(Xu, yn)
        rss_r = np.sum((yn - Xr @ coef_r)**2)
        rss_u = np.sum((yn - Xu @ coef_u)**2)
        df_r = mn - Xr.shape[1]
        df_u = mn - Xu.shape[1]
        if df_u <= 0 or rss_u <= 0:
            continue
        F = ((rss_r - rss_u) / lag) / (rss_u / df_u)
        p = 1 - stats.f.cdf(F, lag, df_u)
        results[lag] = (F, p)
    return results

GRANGER_ALPHA = 0.05
GRANGER_LAG = 3  # use lag-3 (3 months) as primary test

print(f"\nPairwise Granger Causality (lag={GRANGER_LAG}, α={GRANGER_ALPHA})")
print(f"H0: X does NOT Granger-cause Y")
print(f"\n{'X  →  Y':<42} {'F-stat':>8} {'p-value':>10} {'GC?':>8}")
print(f"{'-'*42} {'-'*8} {'-'*10} {'-'*8}")

gc_matrix = np.zeros((N_s, N_s))   # [i,j] = p-value of j Granger-causing i
granger_full = {}
for i, yi_name in enumerate(VARS_STAT):
    for j, xj_name in enumerate(VARS_STAT):
        if i == j:
            continue
        y_ser = X_stat[:, i]
        x_ser = X_stat[:, j]
        res   = granger_test(y_ser, x_ser, max_lag=GRANGER_LAG)
        if GRANGER_LAG in res:
            F, p = res[GRANGER_LAG]
            gc_matrix[i, j] = p
            granger_full[(i, j)] = res
            sig = "✓ Yes" if p < GRANGER_ALPHA else "No"
            print(f"  {xj_name:<18} → {yi_name:<20} {F:>8.3f} {p:>10.4f} {sig:>8}")

# full multi-lag summary
print(f"\nMulti-lag Granger summary (p-values):")
print(f"{'':>22}", end="")
for lag in range(1, GRANGER_LAG+1):
    print(f"lag-{lag}", end="")
print()
for i, yi_name in enumerate(VARS_STAT):
    for j, xj_name in enumerate(VARS_STAT):
        if i == j: continue
        res = granger_full.get((i,j), {})
        if not res: continue
        pvs = [res.get(l, (0,1))[1] for l in range(1, GRANGER_LAG+1)]
        any_sig = any(p < GRANGER_ALPHA for p in pvs)
        if any_sig:
            row = f"{xj_name:>18} → {yi_name:<18}"
            for p in pvs:
                marker = "**" if p < 0.01 else ("*" if p < 0.05 else "  ")
                row += f"{p:.3f}{marker} "
            print(row)

print("PCMCI (PC-based Momentary Conditional Independence for TS)")

"""
  PCMCI Overview:
  PCMCI extends Granger causality by:
  1. Using PC's condition-selection to identify relevant parents per variable.
  2. Testing 'momentary conditional independence' (MCI) to remove
     auto-correlation effects and identify contemporaneous links.
  3. Handles high-dimensional time series with many potential lag parents.

  Two stages:
   Stage 1 (PC^MCI): For each variable Y, find causal parents P(Y) at lags
             1..τ_max using iterative conditional independence tests.
   Stage 2 (MCI):    Test X_{t-τ} → Y_t | P(Y)_t, P(X_{t-1}) for each
             candidate parent, controlling for auto-correlation.
"""

def pcmci(X, var_names, tau_max=4, alpha_pc=0.1, alpha_mci=0.05): # W Claude :) 
    """
    Simplified PCMCI implementation.
    X: (T, N) stationary time series.
    Returns: causal_links dict and p-value matrix.
    """
    T, N = X.shape

    # PC^MCI: parent selection
    print(f"  Stage 1: Parent selection (τ_max={tau_max}, α={alpha_pc})")

    # build lagged data matrix where rows = t, cols = [X_0(t), X_1(t),…, X_0(t-1), X_1(t-1),…]
    # index convention: node i at lag τ → column i + τ*N
    def get_col(X, i, tau, t_start):
        """Return X[:,i] shifted by tau (lag)."""
        if tau == 0:
            return X[t_start:, i]
        return X[t_start-tau: T-tau, i] if t_start >= tau else None

    t_start = tau_max   # first usable time index

    parents = {i: set() for i in range(N)}  # (var_idx, lag) pairs

    # Initialize: all lagged variables as candidate parents (lag 1..tau_max)
    for i in range(N):
        for tau in range(1, tau_max+1):
            parents[i].add((i, tau))    # own lags always start as candidates
            for j in range(N):
                if j != i:
                    parents[i].add((j, tau))

    # Iteratively remove parents using CI tests
    for cond_size in range(0, tau_max * N):
        removed_any = False
        for i in range(N):   # target variable
            for (j, tau) in list(parents[i]):
                # Conditioning set = all current parents except (j,tau)
                cond_parents = list(parents[i] - {(j, tau)})
                if len(cond_parents) < cond_size:
                    continue
                # Build vectors
                y_vec = get_col(X, i, 0, t_start)
                x_vec = get_col(X, j, tau, t_start)
                if y_vec is None or x_vec is None:
                    continue
                # Conditioning matrix
                if cond_size == 0:
                    cond_vecs = []
                else:
                    cond_vecs = [get_col(X, c, ct, t_start)
                                 for c, ct in cond_parents[:cond_size]
                                 if get_col(X, c, ct, t_start) is not None]

                # Fisher-Z on these vectors
                n_obs = len(y_vec)
                if len(cond_vecs) == 0:
                    r = np.corrcoef(y_vec, x_vec)[0, 1]
                else:
                    Z  = np.column_stack(cond_vecs + [np.ones(n_obs)])
                    ry = y_vec - Z @ lstsq(Z, y_vec)[0]
                    rx = x_vec - Z @ lstsq(Z, x_vec)[0]
                    denom = np.sqrt(np.sum(ry**2)*np.sum(rx**2))
                    r = np.dot(ry, rx)/denom if denom > 0 else 0.0

                r  = np.clip(r, -0.9999, 0.9999)
                z  = 0.5 * np.log((1+r)/(1-r))
                se = 1.0 / np.sqrt(max(n_obs - len(cond_vecs) - 3, 1))
                p  = 2*(1 - stats.norm.cdf(abs(z/se)))
                if p >= alpha_pc:
                    parents[i].discard((j, tau))
                    removed_any = True

        if not removed_any:
            break

    print(f"\n  Selected parents per variable:")
    for i, nm in enumerate(var_names):
        par_list = sorted(parents[i], key=lambda x: (x[1], x[0]))
        if par_list:
            par_str = ", ".join(f"{var_names[p]}(t-{t})" for p,t in par_list)
        else:
            par_str = "(none)"
        print(f"    {nm}: {par_str}")

    # ── STAGE 2: MCI test ────────────────────────────────────────────────────
    print(f"\n  Stage 2: MCI tests (α={alpha_mci})")

    # p-value matrix: p_mci[j, tau, i] = p-value for j(t-tau) → i(t)
    mci_pvals   = {}
    causal_links = {i: [] for i in range(N)}

    for i in range(N):
        for j in range(N):
            for tau in range(1, tau_max+1):
                # MCI test: j(t-τ) → i(t) | P(i)\{j(t-τ)}, P(j)(t-1)
                y_vec = get_col(X, i, 0, t_start)
                x_vec = get_col(X, j, tau, t_start)
                if y_vec is None or x_vec is None:
                    continue

                # Build conditioning set
                cond_set_vecs = []
                # Parents of i (excluding the tested link)
                for (pi, pt) in parents[i]:
                    if (pi, pt) != (j, tau):
                        v = get_col(X, pi, pt, t_start)
                        if v is not None:
                            cond_set_vecs.append(v)
                # Parents of j at lag t-1 (i.e., parents of j shifted by 1 more)
                for (pj, pjt) in parents[j]:
                    extra_tau = tau + pjt - 1
                    if extra_tau <= tau_max + 1:
                        v = get_col(X, pj, min(extra_tau, T-t_start-1), t_start)
                        if v is not None and len(v) == len(y_vec):
                            cond_set_vecs.append(v)

                n_obs = len(y_vec)
                if len(cond_set_vecs) == 0:
                    r = np.corrcoef(y_vec, x_vec)[0,1]
                else:
                    # Trim all to same length
                    min_l = min(len(v) for v in cond_set_vecs + [y_vec, x_vec])
                    y_v = y_vec[:min_l]; x_v = x_vec[:min_l]
                    Z   = np.column_stack([v[:min_l] for v in cond_set_vecs]
                                          + [np.ones(min_l)])
                    ry  = y_v - Z @ lstsq(Z, y_v)[0]
                    rx  = x_v - Z @ lstsq(Z, x_v)[0]
                    dn  = np.sqrt(np.sum(ry**2)*np.sum(rx**2))
                    r   = np.dot(ry, rx)/dn if dn > 0 else 0.0
                    n_obs = min_l

                r  = np.clip(r, -0.9999, 0.9999)
                z  = 0.5 * np.log((1+r)/(1-r))
                se = 1.0 / np.sqrt(max(n_obs - len(cond_set_vecs) - 3, 1))
                p  = 2*(1 - stats.norm.cdf(abs(z/se)))
                mci_pvals[(j, tau, i)] = (p, r)

                if p < alpha_mci:
                    causal_links[i].append((j, tau, p, r))

    print(f"\n  Significant MCI causal links (p < {alpha_mci}):")
    print(f"  {'Cause → Effect':<45} {'Lag':>5} {'p-value':>10} {'pcorr':>8}")
    print(f"  {'-'*45} {'-'*5} {'-'*10} {'-'*8}")
    found = False
    for i in range(N):
        for j, tau, p, r in sorted(causal_links[i], key=lambda x: x[2]):
            print(f"  {var_names[j]}(t-{tau}) → {var_names[i]:<30} "
                  f"  {tau:>3}   {p:>10.4f} {r:>8.4f}")
            found = True
    if not found:
        print("  (no links survive MCI test at this threshold)")

    return causal_links, mci_pvals, parents

causal_links, mci_pvals, pc_parents = pcmci(
    X_stat, short_names, tau_max=4, alpha_pc=0.15, alpha_mci=0.05
)

Granger Causality (pairwise + multivariate)

Pairwise Granger Causality (lag=3, α=0.05)
H0: X does NOT Granger-cause Y

X  →  Y                                      F-stat    p-value      GC?
------------------------------------------ -------- ---------- --------
  nontech_ret        → tech_ret                1.641     0.1800       No
  sp500_ret          → tech_ret                1.872     0.1343       No
  d_top10_conc       → tech_ret                0.179     0.9105       No
  d_tech_weight      → tech_ret                1.235     0.2972       No
  tech_ret           → nontech_ret             1.703     0.1665       No
  sp500_ret          → nontech_ret             1.985     0.1162       No
  d_top10_conc       → nontech_ret             1.098     0.3501       No
  d_tech_weight      → nontech_ret             1.291     0.2775       No
  tech_ret           → sp500_ret               0.673     0.5691       No
  nontech_ret        → sp500_ret               1.118     0.3420       No
  d_to

In [9]:
print("Causal Graph Visualizations")

COLOR_TECH = "#2196F3"
COLOR_NONTECH = "#FF9800"
COLOR_MARKET = "#4CAF50"
COLOR_STRUCT = "#9C27B0"

NODE_COLORS = {
    "tech_ret" : COLOR_TECH,
    "nontech_ret" : COLOR_NONTECH,
    "sp500_ret" : COLOR_MARKET,
    "d_conc" : COLOR_STRUCT,
    "d_tw" : COLOR_STRUCT,
    "d_top10_conc" : COLOR_STRUCT,
    "d_tech_weight" : COLOR_STRUCT,
}

def node_color(name):
    for k, c in NODE_COLORS.items():
        if k in name.lower() or name.lower() in k:
            return c
    return "#90A4AE"

FULL_LABELS = {i: name for i, name in enumerate(short_names)}

fig = plt.figure(figsize=(22, 18))
fig.patch.set_facecolor("white") # overall figure background white

fig.suptitle("Causal Discovery: S&P 500 Top-10 Holdings - Tech vs Non-Tech (2000–2025)", fontsize=15, color="black", fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

pos_circle = {
    0: ( 0.0,  1.0), # tech_ret (top)
    1: ( 0.95, 0.31), # nontech_ret (right)
    2: ( 0.59,-0.81), # sp500_ret (bottom-right)
    3: (-0.59,-0.81), # d_conc (bottom-left)
    4: (-0.95, 0.31), # d_tw (left)
}

node_colors_list = [node_color(short_names[i]) for i in range(N_s)]

# Correlation Heatmap
ax_heat = fig.add_subplot(gs[0, 0])
ax_heat.set_facecolor("white")
corr_disp = np.corrcoef(X_stat.T)
im = ax_heat.imshow(corr_disp, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
tick_labels = ["tech\nret", "non-tech\nret", "sp500\nret", "Δconc", "Δt.wt"]
ax_heat.set_xticks(range(N_s))
ax_heat.set_xticklabels(tick_labels, color="black", fontsize=8)
ax_heat.set_yticks(range(N_s))
ax_heat.set_yticklabels(tick_labels, color="black", fontsize=8)
for i in range(N_s):
    for j in range(N_s):
        ax_heat.text(j, i, f"{corr_disp[i,j]:.2f}", ha="center", va="center", fontsize=7, color="black")
cbar = plt.colorbar(im, ax=ax_heat, fraction=0.046)
cbar.ax.yaxis.set_tick_params(color="black")
cbar.outline.set_edgecolor("black")
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color="black")
ax_heat.set_title("Correlation Matrix", color="black", fontsize=10, fontweight="bold")
ax_heat.tick_params(colors="black")
for spine in ax_heat.spines.values():
    spine.set_edgecolor("black")

# Granger Causality Matrix
ax_gc = fig.add_subplot(gs[0, 1])
ax_gc.set_facecolor("white")
gc_disp = gc_matrix.copy()
np.fill_diagonal(gc_disp, 1.0)
im2 = ax_gc.imshow(1 - gc_disp, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax_gc.set_xticks(range(N_s))
ax_gc.set_xticklabels(tick_labels, color="black", fontsize=8)
ax_gc.set_yticks(range(N_s))
ax_gc.set_yticklabels(tick_labels, color="black", fontsize=8)
for i in range(N_s):
    for j in range(N_s):
        if i != j:
            p = gc_disp[i, j]
            marker = "**" if p<0.01 else ("*" if p<0.05 else "")
            ax_gc.text(j, i, f"{p:.2f}{marker}", ha="center", va="center", fontsize=7, color="black")
cbar2 = plt.colorbar(im2, ax=ax_gc, fraction=0.046, label="p-value")
cbar2.outline.set_edgecolor("black")
cbar2.ax.yaxis.set_tick_params(color="black")
plt.setp(plt.getp(cbar2.ax.axes, 'yticklabels'), color="black")
cbar2.set_label("p-value", color="black")
ax_gc.set_title("B. Granger Causality\n(row = effect, col = cause; * p<.05 ** p<.01)", color="black", fontsize=9, fontweight="bold")
ax_gc.tick_params(colors="black")
for spine in ax_gc.spines.values():
    spine.set_edgecolor("black")

# PC Algorithm CPDAG
ax_pc = fig.add_subplot(gs[0, 2])
ax_pc.set_facecolor("white")
ax_pc.set_aspect("equal")

G_draw_pc = nx.DiGraph()
G_draw_pc.add_nodes_from(range(N_s))
edge_styles_pc = {}
for u, v, d in CPDAG.edges(data=True):
    G_draw_pc.add_edge(u, v)
    edge_styles_pc[(u,v)] = d.get("style","directed")
directed_edges = [(u,v) for (u,v),s in edge_styles_pc.items() if s=="directed"]
undirected_edges = [(u,v) for (u,v),s in edge_styles_pc.items() if s=="undirected"]

nx.draw_networkx_nodes(G_draw_pc, pos_circle, ax=ax_pc, node_color=node_colors_list, node_size=1400, alpha=0.9)
nx.draw_networkx_labels(G_draw_pc, pos_circle, ax=ax_pc, 
                        labels={i: short_names[i].replace("_","\n") for i in range(N_s)}, font_size=6.5, 
                        font_color="black", font_weight="bold")
nx.draw_networkx_edges(G_draw_pc, pos_circle, ax=ax_pc, edgelist=directed_edges, 
                       arrows=True, arrowsize=20, edge_color="#1565C0", 
                       width=2.0, connectionstyle="arc3,rad=0.1", node_size=1400)
nx.draw_networkx_edges(G_draw_pc, pos_circle, ax=ax_pc, edgelist=undirected_edges, 
                       arrows=False, edge_color="gray", width=1.5, style="dashed", node_size=1400)
ax_pc.set_title("C. PC Algorithm CPDAG", color="black", fontsize=10, fontweight="bold")
ax_pc.axis("off")

# FCI PAG
ax_fci = fig.add_subplot(gs[1, 0])
ax_fci.set_facecolor("white")
ax_fci.set_aspect("equal")

G_draw_fci = nx.DiGraph()
G_draw_fci.add_nodes_from(range(N_s))
fci_colors = {}
for u, v, d in PAG.edges(data=True):
    G_draw_fci.add_edge(u, v)
    fci_colors[(u,v)] = d.get("etype","undirected")

nx.draw_networkx_nodes(G_draw_fci, pos_circle, ax=ax_fci,
    node_color=node_colors_list, node_size=1400, alpha=0.9)
nx.draw_networkx_labels(G_draw_fci, pos_circle, ax=ax_fci,
    labels={i: short_names[i].replace("_","\n") for i in range(N_s)},
    font_size=6.5, font_color="black", font_weight="bold")
bidir_e = [(u,v) for (u,v),et in fci_colors.items() if et=="bidirected"]
dir_e   = [(u,v) for (u,v),et in fci_colors.items() if et in ("i->j","directed")]
undir_e = [(u,v) for (u,v),et in fci_colors.items() if et=="undirected"]
nx.draw_networkx_edges(G_draw_fci, pos_circle, ax=ax_fci,
    edgelist=dir_e, arrows=True, arrowsize=20, edge_color="#2E7D32",  # darker green
    width=2.0, connectionstyle="arc3,rad=0.1", node_size=1400)
nx.draw_networkx_edges(G_draw_fci, pos_circle, ax=ax_fci,
    edgelist=bidir_e, arrows=True, arrowsize=20, edge_color="#D84315",  # darker orange
    width=2.5, connectionstyle="arc3,rad=0.2", node_size=1400)
nx.draw_networkx_edges(G_draw_fci, pos_circle, ax=ax_fci,
    edgelist=undir_e, arrows=False, edge_color="gray",
    width=1.5, style="dashed", node_size=1400)
ax_fci.set_title("D. FCI PAG\n(orange = bidirected / latent confounder)",
                 color="black", fontsize=9, fontweight="bold")
ax_fci.axis("off")

# PCMCI Time-lag Graph
ax_pcmci = fig.add_subplot(gs[1, 1])
ax_pcmci.set_facecolor("white")
ax_pcmci.set_aspect("equal")

G_pcmci = nx.DiGraph()
G_pcmci.add_nodes_from(range(N_s))
pcmci_edges_by_lag = {1:[], 2:[], 3:[], 4:[]}
for i in range(N_s):
    for j, tau, p, r in causal_links[i]:
        G_pcmci.add_edge(j, i, lag=tau, pval=p, coef=r)
        if tau in pcmci_edges_by_lag:
            pcmci_edges_by_lag[tau].append((j, i))

lag_colors = {1:"#C2185B", 2:"#7B1FA2", 3:"#1565C0", 4:"#00695C"}  # deeper colors
nx.draw_networkx_nodes(G_pcmci, pos_circle, ax=ax_pcmci,
    node_color=node_colors_list, node_size=1400, alpha=0.9)
nx.draw_networkx_labels(G_pcmci, pos_circle, ax=ax_pcmci,
    labels={i: short_names[i].replace("_","\n") for i in range(N_s)},
    font_size=6.5, font_color="black", font_weight="bold")
for lag, ecol in lag_colors.items():
    edges = pcmci_edges_by_lag[lag]
    if edges:
        nx.draw_networkx_edges(G_pcmci, pos_circle, ax=ax_pcmci,
            edgelist=edges, arrows=True, arrowsize=18, edge_color=ecol,
            width=2.5, connectionstyle=f"arc3,rad={0.05*lag}", node_size=1400)

legend_patches = [
    plt.Line2D([0],[0], color=c, linewidth=2.5, label=f"lag-{l}")
    for l, c in lag_colors.items()
]
ax_pcmci.legend(handles=legend_patches, loc="lower right",
                facecolor="white", edgecolor="black",
                labelcolor="black", fontsize=7)
ax_pcmci.set_title("E. PCMCI Causal Time-Lag Graph\n(color = lag depth)",
                   color="black", fontsize=9, fontweight="bold")
ax_pcmci.axis("off")

# Granger Causality Directed Graph
ax_grg = fig.add_subplot(gs[1, 2])
ax_grg.set_facecolor("white")
ax_grg.set_aspect("equal")

G_granger = nx.DiGraph()
G_granger.add_nodes_from(range(N_s))
for i in range(N_s):
    for j in range(N_s):
        if i != j and gc_matrix[i, j] < GRANGER_ALPHA:
            G_granger.add_edge(j, i, pval=gc_matrix[i,j])

nx.draw_networkx_nodes(G_granger, pos_circle, ax=ax_grg,
    node_color=node_colors_list, node_size=1400, alpha=0.9)
nx.draw_networkx_labels(G_granger, pos_circle, ax=ax_grg,
    labels={i: short_names[i].replace("_","\n") for i in range(N_s)},
    font_size=6.5, font_color="black", font_weight="bold")
nx.draw_networkx_edges(G_granger, pos_circle, ax=ax_grg,
    arrows=True, arrowsize=20, edge_color="#F57F17",   # dark yellow
    width=2.0, connectionstyle="arc3,rad=0.12", node_size=1400)
ax_grg.set_title("F. Granger Causality Graph\n(significant links, lag-3, α=0.05)",
                 color="black", fontsize=9, fontweight="bold")
ax_grg.axis("off")

# Node legend (white background)
import matplotlib.patches as mpatches

legend_nodes = [
    mpatches.Patch(facecolor=COLOR_TECH,    label="Tech Returns"),
    mpatches.Patch(facecolor=COLOR_NONTECH, label="Non-Tech Returns"),
    mpatches.Patch(facecolor=COLOR_MARKET,  label="S&P500 Returns"),
    mpatches.Patch(facecolor=COLOR_STRUCT,  label="Structural (weight/conc)"),
]
fig.legend(handles=legend_nodes, loc="lower center", ncol=4,
           facecolor="white", edgecolor="black",
           labelcolor="black", fontsize=9,
           bbox_to_anchor=(0.5, 0.01))

plt.savefig("causal_discovery_sp500.png",
            dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()

# Granger p-value heatmap over lags
fig2, axes2 = plt.subplots(1, N_s, figsize=(18, 4))
fig2.patch.set_facecolor("white")
fig2.suptitle("Granger Causality p-values by Lag (rows = cause, cols = lag)",
              color="black", fontsize=12, fontweight="bold")
for i, ax in enumerate(axes2):
    ax.set_facecolor("white")
    pmat = np.ones((N_s, GRANGER_LAG))
    for j in range(N_s):
        if j == i: continue
        res = granger_full.get((i, j), {})
        for lag in range(1, GRANGER_LAG+1):
            if lag in res:
                pmat[j, lag-1] = res[lag][1]
    im = ax.imshow(pmat, cmap="RdYlGn_r", vmin=0, vmax=0.2, aspect="auto")
    ax.set_xticks(range(GRANGER_LAG))
    ax.set_xticklabels([f"lag-{l}" for l in range(1,GRANGER_LAG+1)],
                       color="black", fontsize=7, rotation=30)
    ax.set_yticks(range(N_s))
    ax.set_yticklabels([short_names[k] for k in range(N_s)],
                       color="black", fontsize=7)
    for r in range(N_s):
        for c in range(GRANGER_LAG):
            ax.text(c, r, f"{pmat[r,c]:.2f}",
                    ha="center", va="center", fontsize=6,
                    color="black")   # always black for light background
    ax.set_title(f"Effect:\n{short_names[i]}", color="black", fontsize=8)
    cbar3 = plt.colorbar(im, ax=ax, fraction=0.046)
    cbar3.outline.set_edgecolor("black")
    cbar3.ax.yaxis.set_tick_params(color="black")
    plt.setp(plt.getp(cbar3.ax.axes, 'yticklabels'), color="black")
    for spine in ax.spines.values():
        spine.set_edgecolor("black")

plt.tight_layout()
plt.savefig("granger_lag_heatmap.png", dpi=150, bbox_inches="tight", facecolor=fig2.get_facecolor())
plt.close()


Causal Graph Visualizations
